In [10]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm         
import sympy as sp
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.tree import plot_tree

In [11]:
calls = pd.read_parquet('data/processed/calls_for_week4.parquet')

y = calls['mid']
X = calls[['underlying_last', 'strike', 'dte', 'r', 'sigma', 'moneyness' ]]


In [13]:
calls = pd.read_parquet('data/processed/calls_for_week4.parquet')
calls['T'] = calls['dte'] / 365
FEATURES = ['underlying_last', 'strike', 'T', 'r', 'sigma', 'moneyness']

train = calls[calls['split'] == 'train'].sample(500_000, random_state=42)
val   = calls[calls['split'] == 'validation'].copy()

rf = RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=5,
                           n_jobs=-1, random_state=42)
rf.fit(train[FEATURES], train['mid'])

gb = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=300,
                                   max_leaf_nodes=31, random_state=42)
gb.fit(train[FEATURES], train['mid'])

val['RF'] = rf.predict(val[FEATURES])
val['GB'] = gb.predict(val[FEATURES])

for name in ['intrinsic', 'BSM', 'RF', 'GB']:
    err = val[name] - val['mid']
    print(f'{name} MAE {err.abs().mean()}   RMSE {np.sqrt((err**2).mean())}')

intrinsic MAE 75.85660762770863   RMSE 124.36594429603649
BSM MAE 19.373187121827574   RMSE 29.948897776854466
RF MAE 10.415607332813652   RMSE 18.56942137735707
GB MAE 11.52108738128046   RMSE 16.586653017895436


In [3]:
plt.figure(figsize=(14, 6))
plot_tree(rf.estimators_[0], feature_names=FEATURES, max_depth=3, filled=True, fontsize=7)
plt.show()

NameError: name 'plt' is not defined